In [1]:
# ============================================================
# One overlay image per patient - no chestwall version
# ------------------------------------------------------------
# 목적:
#   - 지금 돌아가는 no-chest 전처리 결과를 중간 확인
#   - 각 환자마다 대표 slice 1장만 선택
#   - pure_lung / ts_lung_guard / body_guard를 오버레이
#   - chestwall_candidate는 사용하지 않음
#   - 이미 저장된 PNG는 skip
#   - NIfTI 전체 volume을 float32로 올리지 않고 필요한 z slice만 읽음
# ============================================================

from pathlib import Path
import gc

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt


# ============================================================
# 1. CONFIG
# ============================================================

OUT_ROOT = Path(
    r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest"
)

META_CSV = OUT_ROOT / "metadata_slices.csv"

SAVE_DIR = OUT_ROOT / "one_overlay_per_patient_nochest"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = SAVE_DIR / "one_overlay_per_patient_nochest_summary.csv"

HU_MIN = -1000
HU_MAX = 400

# 이미 만든 PNG 스킵
SKIP_EXISTING = True

# 현재 돌아가는 중인 결과를 볼 때 너무 많이 만들기 싫으면 숫자 제한
# None이면 현재 metadata에 있는 모든 완료 환자 처리
MAX_PATIENTS = None
# 예: MAX_PATIENTS = 20


# ============================================================
# 2. 기본 함수
# ============================================================

def safe_name(name: str) -> str:
    return (
        str(name)
        .replace("\\", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("*", "_")
        .replace("?", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("[", "")
        .replace("]", "")
    )


def hu_to_uint8(arr, hu_min=-1000, hu_max=400):
    arr = np.clip(arr.astype(np.float32), hu_min, hu_max)
    arr = (arr - hu_min) / float(hu_max - hu_min + 1e-8)
    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    return arr


def read_nii_slice_array(path: Path, z: int):
    """
    NIfTI에서 필요한 z slice 한 장만 읽음.
    전체 volume을 메모리에 올리지 않아서 가벼움.
    """

    path = Path(path)

    reader = sitk.ImageFileReader()
    reader.SetFileName(str(path))
    reader.ReadImageInformation()

    size = list(reader.GetSize())  # x, y, z

    if len(size) < 3:
        raise RuntimeError(f"3D NIfTI가 아님: {path}, size={size}")

    if z < 0 or z >= size[2]:
        raise RuntimeError(f"z 범위 오류: z={z}, size={size}, path={path}")

    reader.SetExtractIndex([0, 0, int(z)])
    reader.SetExtractSize([int(size[0]), int(size[1]), 1])

    img_slice = reader.Execute()
    arr = sitk.GetArrayFromImage(img_slice)

    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]

    return arr


def path_exists_from_row(row, col_name):
    if col_name not in row.index:
        return False

    value = row[col_name]

    if pd.isna(value) or str(value).strip() == "":
        return False

    return Path(str(value)).exists()


def make_filled_overlay(ct_uint8, pure_mask):
    """
    pure_lung을 노란색으로 채워서 보기 위한 이미지.
    """

    rgb = np.stack([ct_uint8, ct_uint8, ct_uint8], axis=-1).astype(np.uint8)

    pure = pure_mask > 0

    rgb[pure, 0] = 255
    rgb[pure, 1] = 255
    rgb[pure, 2] = 40

    return rgb


# ============================================================
# 3. metadata 로드
# ============================================================

if not META_CSV.exists():
    raise FileNotFoundError(f"metadata_slices.csv 없음: {META_CSV}")

meta_df = pd.read_csv(META_CSV)

if len(meta_df) == 0:
    raise RuntimeError("metadata_slices.csv가 비어 있음. 아직 완료된 환자가 없을 수 있음.")

print("metadata rows:", len(meta_df))
print("patient count in metadata:", meta_df["patient_id"].nunique())


# ============================================================
# 4. 필수 컬럼 확인
# ============================================================

required_cols = [
    "patient_id",
    "slice_index",
    "is_lung_range_slice",
    "pure_lung_area",
    "ts_lung_guard_area",
    "ct_1mm_lps_nii",
    "pure_lung_1mm_nii",
    "ts_lung_guard_1mm_nii",
    "body_guard_1mm_nii",
]

missing_cols = [c for c in required_cols if c not in meta_df.columns]

if len(missing_cols) > 0:
    raise RuntimeError(f"필수 컬럼 누락: {missing_cols}")

print("required columns: OK")


# ============================================================
# 5. 환자별 대표 slice 선택
# ------------------------------------------------------------
# 기준:
#   1순위: lung_range slice
#   2순위: pure_lung_area 가장 큰 slice
#   3순위: ts_lung_guard_area 큰 slice
# ============================================================

summary_rows = []

patient_ids = sorted(meta_df["patient_id"].astype(str).unique())

if MAX_PATIENTS is not None:
    patient_ids = patient_ids[:int(MAX_PATIENTS)]

print("patients to process:", len(patient_ids))

for patient_idx, patient_id in enumerate(patient_ids, start=1):
    patient_df = meta_df[meta_df["patient_id"].astype(str) == patient_id].copy()

    if len(patient_df) == 0:
        continue

    lung_range_df = patient_df[patient_df["is_lung_range_slice"] == 1].copy()

    if len(lung_range_df) > 0:
        candidate_df = lung_range_df
    else:
        candidate_df = patient_df

    candidate_df = candidate_df.sort_values(
        ["pure_lung_area", "ts_lung_guard_area", "slice_index"],
        ascending=[False, False, True],
    )

    row = candidate_df.iloc[0]

    z = int(row["slice_index"])

    ct_path = Path(str(row["ct_1mm_lps_nii"]))
    pure_path = Path(str(row["pure_lung_1mm_nii"]))
    ts_lung_path = Path(str(row["ts_lung_guard_1mm_nii"]))
    body_path = Path(str(row["body_guard_1mm_nii"]))

    out_path = SAVE_DIR / f"{safe_name(patient_id)}_z{z:04d}_overlay_nochest.png"

    if SKIP_EXISTING and out_path.exists():
        summary_rows.append({
            "patient_id": patient_id,
            "selected_slice_index": z,
            "pure_lung_area": int(row["pure_lung_area"]),
            "ts_lung_guard_area": int(row["ts_lung_guard_area"]),
            "saved_png": str(out_path),
            "status": "skipped_existing",
        })
        continue

    missing_paths = []

    for name, p in [
        ("ct", ct_path),
        ("pure_lung", pure_path),
        ("ts_lung_guard", ts_lung_path),
        ("body_guard", body_path),
    ]:
        if not p.exists():
            missing_paths.append(f"{name}: {p}")

    if len(missing_paths) > 0:
        print(f"[SKIP] missing file: {patient_id}")
        for m in missing_paths:
            print(" ", m)

        summary_rows.append({
            "patient_id": patient_id,
            "selected_slice_index": z,
            "pure_lung_area": int(row["pure_lung_area"]),
            "ts_lung_guard_area": int(row["ts_lung_guard_area"]),
            "saved_png": "",
            "status": "missing_file",
            "missing_files": " | ".join(missing_paths),
        })
        continue

    try:
        # ----------------------------------------------------
        # 필요한 slice 한 장만 읽음
        # ----------------------------------------------------
        ct_slice = read_nii_slice_array(ct_path, z)
        pure_slice = read_nii_slice_array(pure_path, z) > 0
        ts_lung_slice = read_nii_slice_array(ts_lung_path, z) > 0
        body_slice = read_nii_slice_array(body_path, z) > 0

        if (
            ct_slice.shape != pure_slice.shape
            or ct_slice.shape != ts_lung_slice.shape
            or ct_slice.shape != body_slice.shape
        ):
            raise RuntimeError(
                f"shape mismatch: "
                f"ct={ct_slice.shape}, "
                f"pure={pure_slice.shape}, "
                f"ts={ts_lung_slice.shape}, "
                f"body={body_slice.shape}"
            )

        ct_uint8 = hu_to_uint8(ct_slice, HU_MIN, HU_MAX)

        pure_rgb = make_filled_overlay(ct_uint8, pure_slice)

        # ----------------------------------------------------
        # 한 환자당 PNG 1장 저장
        # 한 장 안에 3칸
        # ----------------------------------------------------
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # 1) Original CT
        axes[0].imshow(ct_uint8, cmap="gray")
        axes[0].set_title(
            f"Original CT\n{patient_id}\nz={z}",
            fontsize=9,
        )
        axes[0].axis("off")

        # 2) Pure lung filled overlay
        axes[1].imshow(pure_rgb)

        if pure_slice.sum() > 0:
            axes[1].contour(
                pure_slice.astype(np.uint8),
                levels=[0.5],
                colors=["yellow"],
                linewidths=1.2,
            )

        axes[1].set_title(
            f"Pure Lung\narea={int(pure_slice.sum())}",
            fontsize=9,
        )
        axes[1].axis("off")

        # 3) CT + contours
        axes[2].imshow(ct_uint8, cmap="gray")

        # body_guard = lime
        if body_slice.sum() > 0:
            axes[2].contour(
                body_slice.astype(np.uint8),
                levels=[0.5],
                colors=["lime"],
                linewidths=1.0,
            )

        # ts_lung_guard = cyan
        if ts_lung_slice.sum() > 0:
            axes[2].contour(
                ts_lung_slice.astype(np.uint8),
                levels=[0.5],
                colors=["cyan"],
                linewidths=1.2,
            )

        # pure_lung = yellow
        if pure_slice.sum() > 0:
            axes[2].contour(
                pure_slice.astype(np.uint8),
                levels=[0.5],
                colors=["yellow"],
                linewidths=1.2,
            )

        axes[2].set_title(
            "Contours\nlime=body, cyan=TS lung, yellow=pure",
            fontsize=9,
        )
        axes[2].axis("off")

        plt.tight_layout()
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        summary_rows.append({
            "patient_id": patient_id,
            "selected_slice_index": z,
            "pure_lung_area": int(pure_slice.sum()),
            "ts_lung_guard_area": int(ts_lung_slice.sum()),
            "body_guard_area": int(body_slice.sum()),
            "saved_png": str(out_path),
            "status": "saved",
        })

        if patient_idx % 10 == 0 or patient_idx == len(patient_ids):
            print(f"[{patient_idx}/{len(patient_ids)}] saved: {patient_id}")

    except Exception as e:
        print(f"[ERROR] {patient_id}: {e}")

        summary_rows.append({
            "patient_id": patient_id,
            "selected_slice_index": z,
            "pure_lung_area": int(row["pure_lung_area"]) if "pure_lung_area" in row else -1,
            "ts_lung_guard_area": int(row["ts_lung_guard_area"]) if "ts_lung_guard_area" in row else -1,
            "saved_png": "",
            "status": f"error: {str(e)}",
        })

    finally:
        for name in [
            "ct_slice",
            "pure_slice",
            "ts_lung_slice",
            "body_slice",
            "ct_uint8",
            "pure_rgb",
            "fig",
            "axes",
        ]:
            if name in locals():
                del locals()[name]

        gc.collect()


# ============================================================
# 6. summary 저장
# ============================================================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

print("\n========== FINISHED ==========")

if len(summary_df) > 0:
    print("saved image count:", int((summary_df["status"] == "saved").sum()))
    print("skipped existing:", int((summary_df["status"] == "skipped_existing").sum()))
    print("missing file count:", int((summary_df["status"] == "missing_file").sum()))
    print("error count:", int(summary_df["status"].astype(str).str.startswith("error").sum()))

print("save dir:", SAVE_DIR)
print("summary csv:", SUMMARY_CSV)

display(summary_df.head())

metadata rows: 1216
patient count in metadata: 4
required columns: OK
patients to process: 4
[4/4] saved: normal004

========== FINISHED ==========
saved image count: 4
skipped existing: 0
missing file count: 0
error count: 0
save dir: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\one_overlay_per_patient_nochest
summary csv: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\one_overlay_per_patient_nochest\one_overlay_per_patient_nochest_summary.csv


,patient_id,selected_slice_index,pure_lung_area,ts_lung_guard_area,body_guard_area,saved_png,status
0,normal001,100,51245,55856,144726,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,saved
1,normal002,150,52046,56244,123398,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,saved
2,normal003,160,59581,64267,197460,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,saved
3,normal004,116,57725,62808,219614,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,saved
